<a href="https://colab.research.google.com/github/topa1980/Hometasks/blob/main/Hometasks/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F%202%20DGA_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задачи классификации


### 2. Обнаружение DGA-доменов

Эта задача посвящена выявлению алгоритмически сгенерированных доменов второго уровня (SLD) в доменных именах. Ваша задача — построить модель машинного обучения, способную отличать легитимные домены от SLD, сгенерированных с помощью DGA.

**Цель**

Цель — обнаружение алгоритмически сгенерированных доменов второго уровня (SLD), которые часто используются вредоносным ПО для обхода механизмов обнаружения. У нас есть датасет доменных имен с метками: DGA (1) или легитимный (0). Обратите внимание, что в некоторых примерах предоставляется только SLD без домена верхнего уровня (TLD).

Поскольку ложноположительные срабатывания (когда легитимный домен ошибочно классифицируется как DGA) могут приводить к серьезным проблемам, минимизация ложноположительных ошибок важнее, чем минимизация ложноотрицательных.

In [1]:
!pip install tqdm
!pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 2.9 MB/s eta 0:00:00


Скачаем тренировочный датасет по ссылке.

In [2]:
# или скачать файл train.csv с kaggle

!wget -O train.csv "https://drive.usercontent.google.com/download?id=15X1rNwAASXBOd-AGd07t98r2oVy4M_9j&export=download&confirm=t&uuid=7789c270-0113-4dea-967a-6c8290ef1c25"

--2026-02-12 15:39:10--  https://drive.usercontent.google.com/download?id=15X1rNwAASXBOd-AGd07t98r2oVy4M_9j&export=download&confirm=t&uuid=7789c270-0113-4dea-967a-6c8290ef1c25
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 108.177.127.132, 2a00:1450:4013:c07::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|108.177.127.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 331518539 (316M) [application/octet-stream]
Saving to: ‘train.csv’

train.csv           100%[===================>] 316.16M  99.7MB/s    in 3.2s    

2026-02-12 15:39:14 (99.7 MB/s) - ‘train.csv’ saved [331518539/331518539]



In [3]:
import numpy as np
import pandas as pd

from pathlib import Path
from math import log2

from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Посмотрим на данные

In [99]:
data = pd.read_csv("train.csv", nrows=1000)
from sklearn.model_selection import train_test_split
train, test = train_test_split(data, test_size=0.25, random_state=42, stratify= data["label"])



Посмотрим на баланс классов

### Feature engineering

Разобъем данные на train и test
Убери первое разбиение чтобы пройтис по всей выборке

Самая важная честь - придумать признаки, на которых мы будем решать задачу.

In [ ]:
# @title
from collections import Counter
from typing import List, Tuple, Dict, Iterable
import numpy as np

alphabet = 'abcdefghijklmnopqrstuvwxyz'


def ngrams(seq: str, n: int = 3) -> List[str]:
    """Скользящее окно длиной n по символам."""
    if len(seq) < n:
        return []
    return [seq[i:i+n] for i in range(len(seq) - n + 1)]

def ngram_frequencies(seq: str, n: int = 3) -> Counter:
    """Счётчик частот n‑грамм."""
    return Counter(ngrams(seq, n))

def top_k(seq: str, n: int = 3, k: int = 10) -> List[Tuple[str, int]]:
    """k самых частых n‑грамм."""
    return ngram_frequencies(seq, n).most_common(k)

def create_vocab(alphabet: str = 'abcdefghijklmnopqrstuvwxyz', n : int=3):
  from itertools import product
  return {''.join(p): i for i, p in enumerate(product(alphabet, repeat=n))}



vocab3 = create_vocab(alphabet, 3)

sample = "abracadabra"
print("3‑граммы:", ngrams(sample))
print("Частоты:", ngram_frequencies(sample))
print("Топ‑3:", top_k(sample, k=3))
print("Вектор (первые 20 элементов):", [el for el in vectorize(sample, vocab3 ) if el !=0])

whitelist_domains = data.loc[data['label'] == 0, 'domain']
sample = whitelist_domains.str.rsplit(".", n=0).str[0]
print(f"White listing")
white_ngram = Counter([])
#print("3‑граммы:", ngrams(sample))
#print("Частоты:", ngram_frequencies(sample))
#print("Топ‑3:", top_k(sample, k=3))
#print("Вектор (первые 20 элементов):", [el for el in vectorize(sample, vocab3 ) if el !=0])


3‑граммы: ['abr', 'bra', 'rac', 'aca', 'cad', 'ada', 'dab', 'abr', 'bra']
Частоты: Counter({'abr': 2, 'bra': 2, 'rac': 1, 'aca': 1, 'cad': 1, 'ada': 1, 'dab': 1})
Топ‑3: [('abr', 2), ('bra', 2), ('rac', 1)]
Вектор (первые 20 элементов): [np.float32(0.22222222), np.float32(0.11111111), np.float32(0.11111111), np.float32(0.22222222), np.float32(0.11111111), np.float32(0.11111111), np.float32(0.11111111)]
White listing


In [110]:
import re, tldextract, math
from collections import Counter
from itertools import product

import numpy as np

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures

VOWELS = set("aeiou") # множество гласных букв
CONSONANT = set("qzxswdcvfrtgbnhmklp")
DIGIT = set("0123456789")
alphabet = VOWELS and CONSONANT and set("y") #set('abcdefghijklmnopqrstuvwxyz')

def str_domain(arr):
  return arr
def domain_len(arr):
  s = np.asarray(arr, dtype=str).ravel()
  return np.char.str_len(s).reshape(-1, 1)

def digits(arr):
  f = np.vectorize(lambda s: sum( c.isdigit() for c in s ) )
  return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def digit_ratio(arr):
    pat = re.compile(r'\d')
    f = np.vectorize(lambda s: len(pat.findall(s))/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def letters(arr):
  f = np.vectorize(lambda s: sum( c.isalpha() for c in s ) )
  return f(np.asarray(arr, dtype=str).ravel()).reshape(-1, 1)

def letters_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in alphabet for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel()).reshape(-1, 1)

def dash_count(arr):
    return (np.char.count( np.asarray(arr, dtype=str).ravel(), "-" ) ).reshape(-1, 1)

def dash_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in "-" for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def dot_count(arr):
    return (np.char.count( np.asarray(arr, dtype=str).ravel(), "." ) ).reshape(-1, 1)

def dot_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in "." for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def consonant_count(arr):
  f = np.vectorize(lambda s: sum( c in CONSONANT for c in s ) )
  return f(np.asarray(arr, dtype=str).ravel()).reshape(-1, 1)

def consonant_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in CONSONANT for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def vowel_count(arr):
  f = np.vectorize(lambda s: sum( c in VOWELS for c in s ) )
  return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def vowel_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in VOWELS for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def under_char_count(arr):
  f = np.vectorize(lambda s: sum( c in "_" for c in s ) )
  return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def under_char_ratio(arr):
    f = np.vectorize(lambda s: sum(ch in "_" for ch in s)/len(s) if len(s)>0 else 0)
    return f(np.asarray(arr, dtype=str).ravel() ).reshape(-1, 1)

def shannon_entropy(arr):
    def shannon(s):
        if not s: return 0.0
        probs = np.bincount(np.frombuffer(s.encode('utf-8'), dtype=np.uint8))/len(s)
        probs = probs[probs>0]
        return -np.sum(probs*np.log2(probs))
    f = np.vectorize(shannon)
    return f(np.asarray(arr, dtype=str).ravel()).reshape(-1, 1)

def entropy_len_ratio(arr):
  ratio = shannon_entropy(arr)
  lengths = domain_len(arr)
  res = ratio / lengths
  return res.reshape(-1, 1)

def entropy_without_digit(arr):
  _digit_re = re.compile(r'\d')
  def _strip_digits(s: str) -> str:
    return _digit_re.sub('', s)
  strings = np.asarray(arr, dtype=str).ravel()
  no_digits = np.vectorize(_strip_digits, otypes=[str])(strings)
  ent = np.vectorize(shannon_entropy, otypes=[float])(no_digits)
  return ent.reshape(-1, 1)

def entropy_without_dot_value(arr):
  _dot_re = re.compile(r'\.')      # любой символ «.»
  def _strip_dots(s: str) -> str:
    return _dot_re.sub('', s)
  strings = np.asarray(arr, dtype=str).ravel()
  no_digits = np.vectorize(_strip_dots, otypes=[str])(strings)
  ent = np.vectorize(shannon_entropy, otypes=[float])(no_digits)
  return ent.reshape(-1, 1)

      #novel_7_value =  NNGram_with_vowels_ratio(name, 3) #https://habr.com/ru/articles/888234/
    #novel_8_value =  NNGram_with_vowels_ratio(name, 4) #https://habr.com/ru/articles/888234/
    #novel_9_value =  NNGram_with_vowels_ratio(name, 5) #https://habr.com/ru/articles/888234/
#------------------------------
def symbol_run(arr, reg_mask):
    pat = re.compile(reg_mask)
    f = np.vectorize(lambda s: len( max(pat.findall(s) , default = "" ) ) )
    return f(np.asarray(arr, dtype=str).ravel()).reshape(-1, 1)

def max_digit_run(domain):
  return symbol_run(domain, r"\d+")
def max_vowel_run(domain):
  pat = f"[{VOWELS}]+"
  return symbol_run(domain, pat)
def max_consonant_run(domain):
  pat = f"[{CONSONANT}]+"
  return symbol_run(domain, pat)
def max_dash_run(domain):
  pat = f"[-]+"
  return symbol_run(domain, pat)
def max_under_char_run(domain):
  pat = f"[_]"
  return symbol_run(domain, pat)
#----------------------------------------
from sklearn.feature_extraction.text import CountVectorizer

char3 = CountVectorizer(analyzer='char', ngram_range=(3,3), lowercase=True)
char4 = CountVectorizer(analyzer='char', ngram_range=(4,4), lowercase=True)
char4 = CountVectorizer(analyzer='char', ngram_range=(5,5), lowercase=True)

basic_numeric = ColumnTransformer(
    transformers=[
        #('name',   FunctionTransformer(str_domain),   ['domain']),

        ('len',   FunctionTransformer(domain_len),   ['domain']),
        ('digits', FunctionTransformer(digits), ['domain'])
        ,
        ('letters', FunctionTransformer(letters), ['domain']),
        ('dash_count', FunctionTransformer(dash_count), ['domain']),
        ('dot_count', FunctionTransformer(dot_count), ['domain']),
        ('consonant_count', FunctionTransformer(consonant_count), ['domain']),
        ('vowel_count', FunctionTransformer(vowel_count), ['domain']),
        ('under_char_count', FunctionTransformer(under_char_count), ['domain']),

        ('digit_ratio', FunctionTransformer(digit_ratio), ['domain']),
        ('letters_ratio', FunctionTransformer(letters_ratio), ['domain']),
        ('dash_ratio', FunctionTransformer(dash_ratio), ['domain']),
        ('dot_ratio', FunctionTransformer(dot_ratio), ['domain']),
        ('consonant_ratio', FunctionTransformer(consonant_ratio), ['domain']),
        ('vowel_ratio', FunctionTransformer(vowel_ratio), ['domain']),
        ('under_char_ratio', FunctionTransformer(under_char_ratio), ['domain']),

        ('max_digit_run',   FunctionTransformer(max_digit_run),     ['domain']),
        ('max_vowel_run',   FunctionTransformer(max_vowel_run),     ['domain']),
        ('max_consonant_run',   FunctionTransformer(max_consonant_run),     ['domain']),
        ('max_dash_run',   FunctionTransformer(max_dash_run),     ['domain']),
        ('max_under_char_run',   FunctionTransformer(max_under_char_run),     ['domain']),

        ('shannon_entropy',   FunctionTransformer(shannon_entropy),     ['domain']),
        ('entropy_len_ratio',   FunctionTransformer(entropy_len_ratio),     ['domain'])
    ],
    remainder='drop'
)
poly_features=PolynomialFeatures(
    degree = 2,
    include_bias = False,
    interaction_only=False
)
train_without_dot = train['domain'].str.replace(r'\.', '', regex=True)
char3.fit( train_without_dot)

all_features = FeatureUnion([

    ('numeric', basic_numeric)        #  плотные столбцы
    #,('char3' , ('char3', char3) )
])


ft = all_features.fit_transform(train)
print(f"{ft[:2]=}")
print(f"{char3[:3]=}")


def NNGram_with_vowels_ratio(domain_, dimension ):  #Возвращает отношение NN‑грамм с гласными к общему количеству NN‑грамм в переданном доменном имени.
    s = ''.join(ch.lower() for ch in domain_ if ch.isalpha())
    n = len(s)
    if n < dimension:
        return 0.0
    total = n - (dimension - 1 )               # общее количество 3‑грамм (скользящее окно)
    with_vowel = 0
    for i in range(total):
        gram = s[i:i+ dimension]
        if any(ch in VOWELS for ch in gram):
            with_vowel += 1
    return with_vowel / total

def bigram_cv(domain_):
    """Коэффициент вариации частот биграмм"""
    # 1. Извлекаем все биграммы
    bigrams = [domain_[i:i+2] for i in range(len(domain_)-1)]

    # 2. Подсчитываем частоты
    if not bigrams:
        return 0  # для очень коротких строк

    freq = list(Counter(bigrams).values())

    # 3. Вычисляем статистики
    n = len(freq)
    mean = sum(freq) / n

    # 4. Дисперсия и стандартное отклонение
    variance = sum((x - mean) ** 2 for x in freq) / n
    std = math.sqrt(variance)

    # 5. Коэффициент вариации
    # Защита от деления на ноль
    if mean == 0:
        return 0

    return std / mean

def vectorize(seq: str,
              vocab:any,
              n: int = 3,
              alphabet: str = 'abcdefghijklmnopqrstuvwxyz',
              normalize: bool = True) -> np.ndarray:
    """
    Плотный вектор частот всех n‑грамм из заданного алфавита.
    Размер = |alphabet|^n + 1 (OOV).
    """
    vec = np.zeros(len(vocab) + 1, dtype=np.float32)   # последний – OOV
    for gram, count_gramm in ngram_frequencies(seq, n).items():
      vec[vocab.get(gram, -1)] = count_gramm
    if normalize and vec.sum() > 0:
        vec /= vec.sum()
    return vec
#---------------------------------------------

ft[:2]=array([[14.        ,  0.        , 14.        ,  0.        ,  0.        ,
         9.        ,  4.        ,  0.        ,  0.        ,  0.07142857,
         0.        ,  0.        ,  0.64285714,  0.28571429,  0.        ,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ,
         2.98522814,  0.21323058],
       [21.        ,  0.        , 20.        ,  0.        ,  1.        ,
        12.        ,  8.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.04761905,  0.57142857,  0.38095238,  0.        ,
         0.        ,  2.        ,  2.        ,  0.        ,  0.        ,
         3.63041266,  0.17287679]])


TypeError: 'CountVectorizer' object is not subscriptable

Извлечение признаков

In [ ]:
# @title
train["domain"].info()

X_train  = np.array([
    extract_features(str(d))
    for d in tqdm( train["domain"], desc="Extracting train features") # прогресс-бар
])

y_train = train["label"].values

X_test = np.array([
    extract_features(str(d))
    for d in tqdm(test["domain"], desc="Extracting test features")
])

y_test = test["label"].values


In [111]:
# @title
from sklearn.feature_extraction.text import CountVectorizer

whitelist_domains = data.loc[data['label'] == 0, 'domain']
whitelist_domains = whitelist_domains.str.rsplit(".", n=1).str[0]

char3 = CountVectorizer(analyzer='char',      # работаем с символами
                        ngram_range=(3, 3),   # только 3‑gram‑ы
                        lowercase=True)       # регистр не важен

char3.fit(whitelist_domains)                    # обучаем словарь только на whitelist
good_3grams = set(char3.get_feature_names_out())

print(f"{whitelist_domains}")

print(f"Получено {len(good_3grams)} уникальных «хороших» 3‑gram‑ов.")
#---------------------------------------------------------
char4 = CountVectorizer(analyzer='char',      # работаем с символами
                        ngram_range=(4, 4),   # только 3‑gram‑ы
                        lowercase=True)       # регистр не важен

char4.fit(whitelist_domains)                    # обучаем словарь только на whitelist
good_4grams = set(char4.get_feature_names_out())

print(f"Получено {len(good_4grams)} уникальных «хороших» 4‑gram‑ов.")
#------------------------------------------------------
char5 = CountVectorizer(analyzer='char',      # работаем с символами
                        ngram_range=(5, 5),   # только 3‑gram‑ы
                        lowercase=True)       # регистр не важен

char5.fit(whitelist_domains)                    # обучаем словарь только на whitelist
good_5grams = set(char5.get_feature_names_out())

print(f"Получено {len(good_5grams)} уникальных «хороших» 5‑gram‑ов.")




4                 apodoc
6                  esner
9               msp-camp
10      triangleultimate
13       malaysia-hotels
             ...        
994           powergoods
995        weider-bremen
996    fiveguysusedtires
997        tradeprofolex
998         toryaardvark
Name: domain, Length: 570, dtype: object
Получено 2778 уникальных «хороших» 3‑gram‑ов.
Получено 4006 уникальных «хороших» 4‑gram‑ов.
Получено 3917 уникальных «хороших» 5‑gram‑ов.


Перед применением модели давайте масштабируем признаки!

In [101]:
from sklearn.preprocessing import StandardScaler, TargetEncoder, PowerTransformer,MinMaxScaler
model = make_pipeline(
    all_features,
    poly_features,
    StandardScaler(),
    #TargetEncoder(),

    #PowerTransformer(),
    #MinMaxScaler(),
    LogisticRegression( # используем простую модель логрега
        max_iter=1000,
        random_state=0,
        n_jobs=-1,
        class_weight = 'balanced'

    )
)

In [102]:

X_train = train[["domain"]]
y_train = np.ravel(train["label"].values)

X_test = test[["domain"]]
y_test = np.ravel(test["label"].values)

model.fit(X_train, y_train) # обучаем модель

TypeError: All estimators should implement fit and transform. '('char3', CountVectorizer(analyzer='char', ngram_range=(3, 3)))' (type <class 'tuple'>) doesn't

Оценим качество получившейся модели

In [94]:
from sklearn.metrics import fbeta_score

y_test_pred = model.predict(X_test) # предсказываем на валидационной выборке

fbeta = fbeta_score(y_test, y_test_pred, beta=0.5) # вычисляем f_beta скор
print(fbeta)

0.8385156701362909


изменен порог

In [95]:
from sklearn.metrics import classification_report, confusion_matrix

def apply_threshold(p, t=0.75):
    """Возвращает 1, если p >= t, иначе 0."""
    return (p >= t).astype(int)


#print(classification_report(y_test, y_test_pred, digits=4))
#print(confusion_matrix(y_test, y_test_pred))


proba = model.predict_proba(X_test)[:, 1]
y_test_pred_t07 = apply_threshold( proba , 0.7)


print(f"{y_test[:10]=}\n{y_test_pred[:10]=} \n{y_test_pred_t07[:10]=} {proba[:10]}")

print(f" f_beta={fbeta_score(y_test, y_test_pred_t07, beta=0.5)}") # вычисляем f_beta скор



y_test[:10]=array([0, 0, 0, 0, 1, 1, 0, 0, 1, 0])
y_test_pred[:10]=array([0, 0, 0, 0, 1, 1, 0, 0, 1, 0]) 
y_test_pred_t07[:10]=array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0]) [9.80319515e-02 3.60012021e-04 3.20955773e-01 1.50279119e-01
 9.99806166e-01 5.37199102e-01 9.33025980e-03 4.82408386e-05
 9.78842688e-01 4.58026792e-01]
 f_beta=0.8673979859117246


Применим модель LightGBM

In [ ]:
X_test = TargetEncoder(X_test)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, classification_report



lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_test,  label=y_test, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'dart',
    'learning_rate': 0.1,
    'num_leaves': 64,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbosity': 1,
    'seed': 42,
    'is_unbalance': True,
    'device_type ': 'cuda'
}



gbm = lgb.train(params,
                lgb_train,
                num_boost_round=100,
                valid_sets=[lgb_train, lgb_val],
                #early_stopping_rounds=80,
                #verbose_eval=1
                )
proba_gbm = gbm.predict(X_test, num_iteration=gbm.best_iteration)



[LightGBM] [Warning] Unknown parameter: cuda
[LightGBM] [Warning] Unknown parameter: cuda
[LightGBM] [Info] Number of positive: 2364392, number of negative: 2951545
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.910790 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1056
[LightGBM] [Info] Number of data points in the train set: 5315937, number of used features: 14
[LightGBM] [Warning] Unknown parameter: cuda
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.444774 -> initscore=-0.221808
[LightGBM] [Info] Start training from score -0.221808


In [ ]:
from sklearn.metrics import f1_score

print(f" {proba_gbm[:10]}" )


y_test_pred_gbm = apply_threshold( proba_gbm , 0.55)


#y_test_pred_gbm = [ apply_threshold(el, 0.999) for el in y_test_pred_gbm]

print('AUC:', roc_auc_score(y_test, y_test_pred_gbm))
print(f"f_beta={fbeta_score(y_test, y_test_pred_gbm, beta=0.5)}") # вычисляем f_beta скор
print(f"F1 ={f1_score(y_test, y_test_pred_gbm, average='binary')}" )

 [0.30294021 0.17913395 0.7981215  0.20222248 0.96031078 0.19180037
 0.87179701 0.03062314 0.19792916 0.5874181 ]
AUC: 0.8356317399525012
f_beta=0.8425580304923936
F1 =0.8133067300742327


classification_report вывел значения **precision**, **recall**, **F1-score** и **support** для каждого класса:

- **0** — нормальные домены  
- **1** — DGA-домены  

Также выводятся:

- **accuracy** — общая доля правильных предсказаний  

- **macro avg** — среднее арифметическое метрик по классам  

- **weighted avg** — среднее метрик, взвешенное по количеству объектов каждого класса  


### Делаем прогноз на тестовой выборке и отправляем его на Kaggle

In [ ]:
!wget -O test.csv "https://drive.usercontent.google.com/download?id=1MxZ85R9NR1OuaG5XrooF5K0DQaZNI3fN&export=download&confirm=t&uuid=12a40cf0-1911-4d36-9b45-6ff5e60f16df"

--2026-02-09 17:24:54--  https://drive.usercontent.google.com/download?id=1MxZ85R9NR1OuaG5XrooF5K0DQaZNI3fN&export=download&confirm=t&uuid=12a40cf0-1911-4d36-9b45-6ff5e60f16df
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.141.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.141.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 186548519 (178M) [application/octet-stream]
Saving to: ‘test.csv’

test.csv            100%[===================>] 177.91M  62.2MB/s    in 2.9s    

2026-02-09 17:24:58 (62.2 MB/s) - ‘test.csv’ saved [186548519/186548519]



In [ ]:
data_test = pd.read_csv("test.csv") # файл нужно скачать с kaggle

test_features = np.array([
    extract_features(str(d))
    for d in tqdm(data_test["domain"], desc="Extracting test features")
])

Extracting test features: 100%|██████████| 7594197/7594197 [18:21<00:00, 6896.27it/s]


In [ ]:


data_test["label"] = apply_threshold( model.predict_proba(test_features)[:, 1] )
#data_test["label"] = model.predict(test_features).astype(int)

data_test[["id", "label"]].to_csv("submission.csv", index=False) # сохраняем предсказания на тестовой выборке

Далее можно загрузить файл `submission.csv` на страницу соревнования Kaggle  
[**DGA Domain Detection Challenge**](https://www.kaggle.com/t/1f9a87dd384f48449badbe4aac7bb9f1).

Kaggle автоматически оценит ваше решение на **скрытой тестовой выборке** (private leaderboard)  
и вернёт итоговый **score**, отражающий качество предсказаний по метрике соревнования.
